# finetune_augmentation - Custom Data StarDist2D Fine-Tuning

This notebook fine-tunes the fixed base model in `models/stardist_dsb2018_from_scratch/weights_best.h5` on the small `custom_data/` dataset with stronger augmentation.

Use this when the custom dataset has only around 10 images and you want more variation during training.

Expected custom data layout:

```text
custom_data/
  image_name_001/
    Image/          # also supports images/, image/, Images/
      image_name_001.png
    masks/
      mask_001.png
      mask_002.png
```

Images are standardized to `1536 x 1024` pixels (width x height). In NumPy shape terms this is `(1024, 1536)`.


In [ ]:
# Install missing packages only when the current environment does not have them.
import importlib.util
import platform
import subprocess
import sys

required = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'imageio': 'imageio',
    'skimage': 'scikit-image',
    'tensorflow': 'tensorflow',
    'csbdeep': 'csbdeep',
    'stardist': 'stardist',
}

IS_LINUX = sys.platform.startswith('linux')
IS_MAC_ARM = sys.platform == 'darwin' and platform.machine() == 'arm64'
if IS_MAC_ARM:
    required['tensorflow_metal'] = 'tensorflow-metal'
if IS_LINUX:
    required['nvidia.cudnn'] = 'tensorflow[and-cuda]'
    required['nvidia.cublas'] = 'tensorflow[and-cuda]'

missing = sorted(set(pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None))
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are already installed.')


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import random
import sys

import numpy as np

# Make matplotlib cache writable in restricted WSL/remote environments.
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

# WSL/Linux: expose CUDA/cuDNN pip libraries before importing TensorFlow.
def add_pip_nvidia_libs_to_ld_path():
    if not sys.platform.startswith('linux'):
        return []
    site_packages = Path(sys.prefix) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
    nvidia_dir = site_packages / 'nvidia'
    if not nvidia_dir.exists():
        return []
    lib_dirs = sorted(str(path) for path in nvidia_dir.glob('*/lib') if path.is_dir())
    if not lib_dirs:
        return []
    current = os.environ.get('LD_LIBRARY_PATH', '')
    existing = [path for path in current.split(':') if path]
    merged = lib_dirs + [path for path in existing if path not in lib_dirs]
    os.environ['LD_LIBRARY_PATH'] = ':'.join(merged)
    return lib_dirs

CUDA_PIP_LIB_DIRS = add_pip_nvidia_libs_to_ld_path()
if CUDA_PIP_LIB_DIRS:
    print('Added CUDA pip library dirs:', len(CUDA_PIP_LIB_DIRS))

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from imageio.v3 import imread
from skimage.color import rgb2gray, rgba2rgb
from skimage.segmentation import find_boundaries
from skimage.transform import AffineTransform, resize, warp
from scipy.ndimage import gaussian_filter

import tensorflow as tf
from csbdeep.utils import normalize
from stardist import fill_label_holes, random_label_cmap
from stardist.models import Config2D, StarDist2D

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)


In [ ]:
# Device selection. Use CUDA by default in this WSL workstation.
DEVICE_PREFERENCE = 'cuda'  # 'cuda', 'mps', 'auto', 'cpu'

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print(f'Memory growth could not be set for {gpu}: {exc}')

logical_gpus = tf.config.list_logical_devices('GPU')
pluggable_devices = tf.config.list_logical_devices('PluggableDevice')

if DEVICE_PREFERENCE == 'mps':
    if pluggable_devices:
        DEVICE_NAME = pluggable_devices[0].name
        DEVICE_KIND = 'Apple Metal/MPS PluggableDevice'
    elif sys.platform == 'darwin' and logical_gpus:
        DEVICE_NAME = logical_gpus[0].name
        DEVICE_KIND = 'Apple Metal/MPS via TensorFlow GPU device'
    else:
        raise RuntimeError('MPS device를 찾지 못했습니다. Windows/WSL에서는 DEVICE_PREFERENCE를 cuda 또는 auto로 바꾸세요.')
elif DEVICE_PREFERENCE == 'cuda':
    if logical_gpus:
        DEVICE_NAME = logical_gpus[0].name
        DEVICE_KIND = 'CUDA/TensorFlow GPU'
    else:
        raise RuntimeError('CUDA GPU를 찾지 못했습니다. DEVICE_PREFERENCE를 auto 또는 cpu로 바꾸거나 NVIDIA/CUDA 설정을 확인하세요.')
elif DEVICE_PREFERENCE == 'cpu':
    DEVICE_NAME = '/CPU:0'
    DEVICE_KIND = 'CPU'
else:
    if pluggable_devices:
        DEVICE_NAME = pluggable_devices[0].name
        DEVICE_KIND = 'Apple Metal/MPS PluggableDevice'
    elif logical_gpus:
        DEVICE_NAME = logical_gpus[0].name
        DEVICE_KIND = 'CUDA/TensorFlow GPU'
    else:
        DEVICE_NAME = '/CPU:0'
        DEVICE_KIND = 'CPU'

print('Physical GPUs:', physical_gpus)
print('Logical GPUs:', logical_gpus)
print('Pluggable devices:', pluggable_devices)
print('Selected device:', DEVICE_NAME, '|', DEVICE_KIND)


In [ ]:
# ===== User settings =====
PROJECT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = [
    PROJECT_DIR,
    *PROJECT_DIR.parents,
    Path('/home/ksc/projects/stardistTest'),
    PROJECT_DIR / 'stardistTest',
]

PROJECT_DIR = None
for base in PROJECT_CANDIDATES:
    if (base / 'data-science-bowl-2018').exists() or (base / 'custom_data').exists() or (base / 'models').exists():
        PROJECT_DIR = base.resolve()
        break

if PROJECT_DIR is None:
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. stardistTest 폴더에서 이 노트북을 실행하세요.')

CUSTOM_DATA_DIR = PROJECT_DIR / 'custom_data'
CUSTOM_DATA_DIR.mkdir(exist_ok=True)

BASE_MODEL_DIR = PROJECT_DIR / 'models' / 'stardist_dsb2018_from_scratch'
BASE_WEIGHTS_PATH = BASE_MODEL_DIR / 'weights_best.h5'
MODEL_BASEDIR = PROJECT_DIR / 'models'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
MODEL_NAME = f'stardist_dsb2018_finetune_augmentation_{RUN_ID}'

TARGET_WIDTH = 1536
TARGET_HEIGHT = 1024
TARGET_SHAPE = (TARGET_HEIGHT, TARGET_WIDTH)

PATCH_SIZE = (256, 256)
BATCH_SIZE = 1
EPOCHS = 40
LEARNING_RATE = 1e-5
VALIDATION_SPLIT = 0.2
UNFREEZE_LAST_N_LAYERS = 6
MIN_STEPS_PER_EPOCH = 80
STEPS_MULTIPLIER = 16
FAST_DEV_RUN = False
MAX_TRAIN_IMAGES = 8 if FAST_DEV_RUN else None

# Stronger augmentation settings for very small custom datasets.
AUGMENT_ROTATE_DEGREES = 12
AUGMENT_TRANSLATE_PIXELS = 18
AUGMENT_SCALE_RANGE = (0.92, 1.08)
AUGMENT_NOISE_STD_RANGE = (0.0, 0.025)
AUGMENT_BLUR_SIGMA_RANGE = (0.0, 0.6)
AUGMENT_GAMMA_RANGE = (0.85, 1.2)

if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(f'Base model directory not found: {BASE_MODEL_DIR}')
if not BASE_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f'Base weights not found: {BASE_WEIGHTS_PATH}')

print('Project dir:', PROJECT_DIR)
print('Custom data dir:', CUSTOM_DATA_DIR)
print('Base weights:', BASE_WEIGHTS_PATH)
print('Fine-tuned model output:', MODEL_BASEDIR / MODEL_NAME)
print('Target image shape:', TARGET_SHAPE)


In [ ]:
EXPECTED_LAYOUT = """
custom_data/
  image_name_001/
    images/
      image_name_001.png
    masks/
      mask_001.png
      mask_002.png
"""


def find_pngs_in_first_existing_dir(sample_dir: Path, candidates):
    for dirname in candidates:
        folder = sample_dir / dirname
        files = sorted(folder.glob('*.png')) if folder.exists() else []
        if files:
            return files, folder
    return [], sample_dir / candidates[0]


def get_image_files(sample_dir: Path):
    return find_pngs_in_first_existing_dir(sample_dir, ['images', 'Image', 'image', 'Images'])


def get_mask_files(sample_dir: Path):
    return find_pngs_in_first_existing_dir(sample_dir, ['masks', 'Masks'])


def list_custom_image_ids(custom_dir: Path):
    ids = []
    skipped = []
    for p in sorted(custom_dir.iterdir()):
        if not p.is_dir():
            continue
        image_files, image_dir = get_image_files(p)
        mask_files, mask_dir = get_mask_files(p)
        if image_files and mask_files:
            ids.append(p.name)
        else:
            skipped.append((p.name, len(image_files), len(mask_files), image_dir.name, mask_dir.name))
    return ids, skipped

image_ids, skipped_samples = list_custom_image_ids(CUSTOM_DATA_DIR)
if MAX_TRAIN_IMAGES is not None:
    image_ids = image_ids[:MAX_TRAIN_IMAGES]

if skipped_samples:
    print('Skipped folders without both images/*.png and masks/*.png:')
    for sample_id, n_images, n_masks, image_dir, mask_dir in skipped_samples[:20]:
        print(f'  {sample_id}: {image_dir}={n_images}, {mask_dir}={n_masks}')
    if len(skipped_samples) > 20:
        print('  ...')

print(f'Usable custom images: {len(image_ids)}')
print('First IDs:', image_ids[:5])

if len(image_ids) == 0:
    raise ValueError(
        f'No usable custom samples found in {CUSTOM_DATA_DIR}.\n'
        f'Expected layout:\n{EXPECTED_LAYOUT}'
    )
if len(image_ids) < 2:
    raise ValueError('At least 2 usable custom samples are required so one image can be held out for validation.')


In [ ]:
def read_grayscale01(path: Path) -> np.ndarray:
    """Read PNG/RGBA/RGB/grayscale image and return float32 grayscale in [0, 1]."""
    img = imread(path)
    if img.ndim == 2:
        gray = img.astype(np.float32)
    elif img.ndim == 3 and img.shape[-1] == 4:
        rgb = rgba2rgb(img.astype(np.float32) / 255.0)
        return rgb2gray(rgb).astype(np.float32)
    elif img.ndim == 3 and img.shape[-1] >= 3:
        rgb = img[..., :3].astype(np.float32)
        if rgb.max() > 1:
            rgb = rgb / 255.0
        return rgb2gray(rgb).astype(np.float32)
    else:
        raise ValueError(f'Unsupported image shape {img.shape} for {path}')

    if gray.max() > 1:
        gray = gray / np.iinfo(img.dtype).max if np.issubdtype(img.dtype, np.integer) else gray / gray.max()
    return gray.astype(np.float32)


def resize_image_to_target(image: np.ndarray) -> np.ndarray:
    if image.shape == TARGET_SHAPE:
        return image.astype(np.float32)
    return resize(
        image,
        TARGET_SHAPE,
        order=1,
        mode='reflect',
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)


def read_binary_mask(path: Path) -> np.ndarray:
    mask = imread(path)
    if mask.ndim == 3:
        mask = mask[..., 0]
    if mask.shape != TARGET_SHAPE:
        mask = resize(
            mask,
            TARGET_SHAPE,
            order=0,
            mode='edge',
            preserve_range=True,
            anti_aliasing=False,
        )
    return mask > 0


def load_instance_label(image_id: str) -> np.ndarray:
    sample_dir = CUSTOM_DATA_DIR / image_id
    image_path = get_image_files(sample_dir)[0][0]
    image_shape = resize_image_to_target(read_grayscale01(image_path)).shape
    if image_shape != TARGET_SHAPE:
        raise ValueError(f'Image preprocessing failed for {image_id}: {image_shape} != {TARGET_SHAPE}')

    label = np.zeros(TARGET_SHAPE, dtype=np.uint16)
    mask_paths = get_mask_files(sample_dir)[0]
    for instance_id, mask_path in enumerate(mask_paths, start=1):
        mask = read_binary_mask(mask_path)
        if mask.shape != TARGET_SHAPE:
            raise ValueError(f'Mask preprocessing failed for {image_id}: {mask.shape} != {TARGET_SHAPE}')
        label[mask] = instance_id
    return fill_label_holes(label)


def load_image_and_label(image_id: str):
    sample_dir = CUSTOM_DATA_DIR / image_id
    image_path = get_image_files(sample_dir)[0][0]
    image = resize_image_to_target(read_grayscale01(image_path))
    label = load_instance_label(image_id)
    return image, label


In [ ]:
# Check one sample before loading the full custom dataset.
sample_id = image_ids[0]
sample_x, sample_y = load_image_and_label(sample_id)
print('Sample ID:', sample_id)
print('image:', sample_x.shape, sample_x.dtype, float(sample_x.min()), float(sample_x.max()))
print('label:', sample_y.shape, sample_y.dtype, 'instances:', int(sample_y.max()))

plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1)
plt.imshow(sample_x, cmap='gray')
plt.title('image grayscale')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(sample_y, cmap=random_label_cmap())
plt.title('merged instance labels')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(sample_x, cmap='gray')
sample_boundaries = find_boundaries(sample_y)
plt.imshow(np.ma.masked_where(~sample_boundaries, sample_boundaries), cmap='autumn', alpha=0.9)
plt.title('GT boundaries overlay')
plt.axis('off')
plt.show()


In [ ]:
X, Y = [], []
for image_id in tqdm(image_ids, desc='Loading custom images and masks'):
    x, y = load_image_and_label(image_id)
    X.append(x)
    Y.append(y)

X = [normalize(x, 1, 99.8, axis=(0, 1)).astype(np.float32) for x in X]
Y = [y.astype(np.uint16) for y in Y]

assert len(X) == len(Y) > 0
assert all(x.shape == TARGET_SHAPE for x in X)
assert all(x.shape == y.shape for x, y in zip(X, Y))
print('Loaded:', len(X), 'images')
print('All image shapes:', sorted(set(x.shape for x in X)))
print('Instance count examples:', [int(y.max()) for y in Y[:5]])


In [ ]:
# Deterministic train/validation split.
indices = np.arange(len(X))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_val = max(1, int(round(len(indices) * VALIDATION_SPLIT)))
n_val = min(n_val, len(indices) - 1)
val_idx = indices[:n_val]
train_idx = indices[n_val:]

X_train = [X[i] for i in train_idx]
Y_train = [Y[i] for i in train_idx]
X_val = [X[i] for i in val_idx]
Y_val = [Y[i] for i in val_idx]

STEPS_PER_EPOCH = max(MIN_STEPS_PER_EPOCH, len(X_train) * STEPS_MULTIPLIER)

print('Train:', len(X_train), 'Validation:', len(X_val))
print('Steps per epoch:', STEPS_PER_EPOCH)


In [ ]:
def random_fliprot(x, y):
    k = np.random.randint(4)
    x = np.rot90(x, k)
    y = np.rot90(y, k)
    if np.random.rand() > 0.5:
        x = np.fliplr(x)
        y = np.fliplr(y)
    if np.random.rand() > 0.5:
        x = np.flipud(x)
        y = np.flipud(y)
    return x, y


def random_affine(x, y):
    """Apply the same mild affine transform to image and instance labels."""
    h, w = x.shape[:2]
    angle = np.deg2rad(np.random.uniform(-AUGMENT_ROTATE_DEGREES, AUGMENT_ROTATE_DEGREES))
    scale = np.random.uniform(*AUGMENT_SCALE_RANGE)
    tx = np.random.uniform(-AUGMENT_TRANSLATE_PIXELS, AUGMENT_TRANSLATE_PIXELS)
    ty = np.random.uniform(-AUGMENT_TRANSLATE_PIXELS, AUGMENT_TRANSLATE_PIXELS)

    center = np.array([w / 2.0, h / 2.0])
    tform = (
        AffineTransform(translation=-center)
        + AffineTransform(scale=(scale, scale), rotation=angle)
        + AffineTransform(translation=center + np.array([tx, ty]))
    )

    x_aug = warp(
        x,
        inverse_map=tform.inverse,
        order=1,
        mode='reflect',
        preserve_range=True,
    ).astype(np.float32)
    y_aug = warp(
        y,
        inverse_map=tform.inverse,
        order=0,
        mode='constant',
        cval=0,
        preserve_range=True,
    ).astype(y.dtype)
    return x_aug, y_aug


def random_intensity(x):
    # Brightness/contrast shift.
    scale = np.random.uniform(0.85, 1.18)
    shift = np.random.uniform(-0.08, 0.08)
    x = x * scale + shift

    # Gamma augmentation. Input is normalized to [0, 1]-ish, so clip first.
    x = np.clip(x, 0, 1)
    gamma = np.random.uniform(*AUGMENT_GAMMA_RANGE)
    x = np.power(x, gamma)

    # Occasional blur and gaussian noise.
    sigma = np.random.uniform(*AUGMENT_BLUR_SIGMA_RANGE)
    if sigma > 0.05 and np.random.rand() < 0.45:
        x = gaussian_filter(x, sigma=sigma)

    noise_std = np.random.uniform(*AUGMENT_NOISE_STD_RANGE)
    if noise_std > 0 and np.random.rand() < 0.65:
        x = x + np.random.normal(0, noise_std, size=x.shape).astype(np.float32)

    return np.clip(x, 0, 1).astype(np.float32)


def augmenter(x, y):
    x, y = random_fliprot(x, y)
    if np.random.rand() < 0.75:
        x, y = random_affine(x, y)
    x = random_intensity(x)
    return x.astype(np.float32), y.astype(np.uint16)


In [ ]:
# Preview random augmentations on the first training sample.
preview_x = X_train[0]
preview_y = Y_train[0]
plt.figure(figsize=(16, 8))
for i in range(6):
    ax_x = plt.subplot(2, 6, i + 1)
    aug_x, aug_y = augmenter(preview_x.copy(), preview_y.copy())
    ax_x.imshow(aug_x, cmap='gray')
    ax_x.set_title(f'aug image {i + 1}')
    ax_x.axis('off')

    ax_y = plt.subplot(2, 6, i + 7)
    ax_y.imshow(aug_x, cmap='gray')
    boundaries = find_boundaries(aug_y)
    ax_y.imshow(np.ma.masked_where(~boundaries, boundaries), cmap='autumn', alpha=0.9)
    ax_y.set_title(f'GT overlay {int(aug_y.max())}')
    ax_y.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
conf = Config2D(
    n_rays=32,
    grid=(1, 1),
    n_channel_in=1,
    train_patch_size=PATCH_SIZE,
    train_batch_size=BATCH_SIZE,
    train_epochs=EPOCHS,
    train_steps_per_epoch=STEPS_PER_EPOCH,
    train_learning_rate=LEARNING_RATE,
    unet_n_depth=3,
    unet_n_filter_base=32,
    unet_n_conv_per_depth=2,
)
print(conf)

with tf.device(DEVICE_NAME):
    model = StarDist2D(conf, name=MODEL_NAME, basedir=str(MODEL_BASEDIR))
    model.keras_model.load_weights(str(BASE_WEIGHTS_PATH))

print('Loaded base weights:', BASE_WEIGHTS_PATH)
print('Fine-tuned checkpoints will be written to:', model.logdir)


In [ ]:
def freeze_for_small_dataset(keras_model, unfreeze_last_n_layers: int):
    for layer in keras_model.layers:
        layer.trainable = False

    if unfreeze_last_n_layers <= 0:
        trainable_slice = []
    else:
        trainable_slice = keras_model.layers[-unfreeze_last_n_layers:]

    for layer in trainable_slice:
        layer.trainable = True

    trainable_layers = [layer.name for layer in keras_model.layers if layer.trainable]
    frozen_layers = [layer.name for layer in keras_model.layers if not layer.trainable]
    print('Frozen layers:', len(frozen_layers))
    print('Trainable layers:', len(trainable_layers))
    print('Trainable tail:', trainable_layers)


freeze_for_small_dataset(model.keras_model, UNFREEZE_LAST_N_LAYERS)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.prepare_for_training(optimizer=optimizer)
model.keras_model.summary()


In [ ]:
# Fine-tune from the fixed DSB base weights.
with tf.device(DEVICE_NAME):
    history = model.train(
        X_train,
        Y_train,
        validation_data=(X_val, Y_val),
        augmenter=augmenter,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        seed=SEED,
    )


In [ ]:
# Optimize probability/NMS thresholds on the held-out validation samples.
model.optimize_thresholds(X_val, Y_val)
print('Optimized thresholds:', model.thresholds)
print('Saved fine-tuned model:', model.logdir)


In [ ]:
# Training curves.
hist = history.history
plt.figure(figsize=(12, 4))
for key in ['loss', 'prob_loss', 'dist_loss']:
    if key in hist:
        plt.plot(hist[key], label=key)
    val_key = 'val_' + key
    if val_key in hist:
        plt.plot(hist[val_key], label=val_key)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Validation prediction sanity check.
sample_i = 0
x = X_val[sample_i]
y_true = Y_val[sample_i]
y_pred, details = model.predict_instances(x)

plt.figure(figsize=(16, 5))
plt.subplot(1, 3, 1)
plt.imshow(x, cmap='gray')
plt.title('validation image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(y_true, cmap=random_label_cmap())
plt.title(f'GT instances: {int(y_true.max())}')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(x, cmap='gray')
pred_boundaries = find_boundaries(y_pred)
plt.imshow(np.ma.masked_where(~pred_boundaries, pred_boundaries), cmap='autumn', alpha=0.9)
plt.title(f'Prediction instances: {int(y_pred.max())}')
plt.axis('off')
plt.show()
